## New File for Pre-training

In [ ]:
import os
import copy
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from typing import Any
import math
from sklearn.preprocessing import StandardScaler

In [ ]:
class ToTensor:
    def __call__(self, sample):
        return torch.tensor(sample, dtype=torch.float32)

In [ ]:
class Normalize:
    def __call__(self, sample):
        mean = sample.mean(dim=1, keepdim=True)
        std = sample.std(dim=1, keepdim=True) + 1e-6
        return (sample - mean) / std

In [ ]:
class GaussianNoise:
    def __init__(self, mean=0., std=1.):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

In [ ]:
import numpy as np
import torch

class MultiBlockMask:
    def __init__(self, num_masks=3, mask_ratio=0.05):
        """
        Random block masking for 1D signals.
        Args:
            num_masks (int): number of masked regions
            mask_ratio (float): fraction of signal length per block
        """
        self.num_masks = num_masks
        self.mask_ratio = mask_ratio

    def __call__(self, signal):
        """
        Apply masking to the signal.
        Args:
            signal (np.ndarray or torch.Tensor): input 1D signal
        Returns:
            same type as input (masked signal)
        """
        is_tensor = isinstance(signal, torch.Tensor)

        if is_tensor:
            signal_np = signal.detach().cpu().numpy()
        else:
            signal_np = signal.copy()

        L = len(signal_np)
        for _ in range(self.num_masks):
            mask_len = int(L * self.mask_ratio)
            start = np.random.randint(0, L - mask_len)
            signal_np[start:start+mask_len] = 0

        if is_tensor:
            return torch.tensor(signal_np, dtype=signal.dtype, device=signal.device)
        return signal_np

In [ ]:
class NoiseMask:
    def __init__(self, mask_ratio=0.1):
        """
        Replace a random block of the signal with Gaussian noise.
        Args:
            mask_ratio (float): fraction of signal length to mask
        """
        self.mask_ratio = mask_ratio

    def __call__(self, signal):
        """
        Apply noise masking to the signal.
        Args:
            signal (torch.Tensor): input 1D tensor
        Returns:
            torch.Tensor: masked tensor
        """
        L = signal.shape[-1]
        mask_len = int(L * self.mask_ratio)
        start = np.random.randint(0, L - mask_len)
        signal_masked = signal.clone()
        mean = signal.mean().item()
        std = signal.std().item()
        noise = torch.normal(mean=mean, std=std, size=(mask_len,), device=signal.device)
        signal_masked[..., start:start+mask_len] = noise
        return signal_masked

In [ ]:
class DatasetWrapper(Dataset):
    def __init__(self, data_path, noise_std=0.05):
        self.X = np.load(data_path)  
        if self.X.ndim == 2:  
            self.X = self.X[:, np.newaxis, :]
        self.to_tensor = ToTensor()
        self.normalize = Normalize()
        self.gaussian_noise = GaussianNoise(std=noise_std)
        self.multi_block_mask = MultiBlockMask(num_masks=3, mask_ratio=0.05)
        self.noise_mask = NoiseMask(mask_ratio=0.1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        sample = self.X[idx]  
        sample = self.to_tensor(sample)  
        norm_sample = sample  
        masked_sample = self.multi_block_mask(norm_sample.clone())
        combined_sample = torch.stack([norm_sample,  masked_sample], dim=-1)  
        return combined_sample.float()

In [ ]:
clamp_val = 2.0

In [ ]:
class LinearEmbed(nn.Module):

    def __init__(self, num_lead: int, chunk_len: int, d_model: int) -> None:
        super(LinearEmbed, self).__init__()
        self.num_lead = num_lead
        self.chunk_len = chunk_len
        chunk_dim = self.num_lead * self.chunk_len
        self.embed = nn.Linear(chunk_dim, d_model)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x (torch.Tensor): Tensor of size (batch_size, num_lead, seqlen).
        Retunp.load(self.data_path)rns:
            feat (torch.Tensor): Tensor of size (batch_size, num_chunks, d_model).
        """
        assert(x.size(1) == self.num_lead)
        assert(x.size(2) % self.chunk_len == 0)

        bs = x.size(0)
        num_chunks = x.size(2) // self.chunk_len
        # batch_size, num_lead, num_chunks, chunk_len
        x = torch.reshape(x, (bs, self.num_lead, num_chunks, self.chunk_len))
        x = x.permute(0, 2, 1, 3)
        # batch_size, num_chunks, num_lead * chunk_len
        x = torch.reshape(x, (bs, num_chunks, -1))
        feat = self.embed(x)
        return feat

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=3):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, channels, seq_len)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)  # (batch, 2, seq_len)
        attn = self.sigmoid(self.conv1(concat))        # (batch, 1, seq_len)
        return x * attn  

In [ ]:
class ConvEmbed(nn.Module):
    def __init__(self, num_lead: int, d_model: int) -> None:
        super(ConvEmbed, self).__init__()
        self.num_lead = num_lead
        self.conv1 = nn.Conv1d(num_lead, 128, kernel_size=14, stride=3, padding=2)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=14, stride=3, padding=0)
        self.conv3 = nn.Conv1d(256, 256, kernel_size=10, stride=2, padding=0)
        self.conv4 = nn.Conv1d(256, 256, kernel_size=10, stride=1, padding=0)
        self.attn = SpatialAttention()
        self.dense = nn.Linear(256, d_model)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x (torch.Tensor): (batch_size, num_lead, seqlen)
        Returns:
            feat (torch.Tensor): (batch_size, num_steps, d_model)
        """
        feat = self.conv1(x)
        feat = self.conv2(feat)
        feat = self.conv3(feat)
        feat = self.conv4(feat)
        feat = self.attn(feat)
        feat = feat.permute(0, 2, 1)  # (B, T, C)
        feat = self.dense(feat)       # (B, T, d_model)
        return feat

In [ ]:
class PositionalEncoding(nn.Module):
    """
    From `https://pytorch.org/tutorials/beginner/transformer_tutorial.html`
    """
    def __init__(
        self,
        d_model: int,
        dropout: float=0.1,
        max_len: int=1648
    ) -> None:
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor):
        """
        Add positional encoding.
        Args:
            x (torch.Tensor): Tensor of size (batch_size, num_steps, d_model).
        Returns:
            x (torch.Tensor): Tensor of size (batch_size, num_steps, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

In [ ]:
import math
from enum import Enum
import torch
import torch.nn as nn


class MixingMatrixInit(Enum):
    CONCATENATE = 1
    ALL_ONES = 2
    UNIFORM = 3


class CollaborativeAttention(nn.Module):
    def __init__(
        self,
        dim_input: int,
        dim_value_all: int,
        dim_key_query_all: int,
        dim_output: int,
        num_attention_heads: int,
        output_attentions: bool,
        attention_probs_dropout_prob: float,
        use_dense_layer: bool,
        use_layer_norm: bool,
        mixing_initialization: MixingMatrixInit = MixingMatrixInit.UNIFORM,
    ):
        super().__init__()

        if dim_value_all % num_attention_heads != 0:
            raise ValueError(
                "Value dimension ({}) should be divisible by number of heads ({})".format(
                    dim_value_all, num_attention_heads
                )
            )

        if not use_dense_layer and dim_value_all != dim_output:
            raise ValueError(
                "Output dimension ({}) should be equal to value dimension ({}) if no dense layer is used".format(
                    dim_output, dim_value_all
                )
            )
        # save args
        self.dim_input = dim_input
        self.dim_value_all = dim_value_all
        self.dim_key_query_all = dim_key_query_all
        self.dim_output = dim_output
        self.num_attention_heads = num_attention_heads
        self.output_attentions = output_attentions
        self.attention_probs_dropout_prob = attention_probs_dropout_prob
        self.mixing_initialization = mixing_initialization
        self.use_dense_layer = use_dense_layer
        self.use_layer_norm = use_layer_norm

        self.dim_value_per_head = dim_value_all // num_attention_heads
        self.attention_head_size = (
            dim_key_query_all / num_attention_heads
        )  # does not have to be integer

        # intialize parameters
        self.query = nn.Linear(dim_input, dim_key_query_all, bias=False)
        self.key = nn.Linear(dim_input, dim_key_query_all, bias=False)
        self.content_bias = nn.Linear(dim_input, num_attention_heads, bias=False)
        self.value = nn.Linear(dim_input, dim_value_all)
        # del self.m_t
        # del self.m_c
        
        self.m_t = self.init_mixing_matrix()
        self.m_c = self.init_mixing_matrix()

        self.dense = (
            nn.Linear(dim_value_all, dim_output) if use_dense_layer else nn.Sequential()
        )
        self.dropout = nn.Dropout(attention_probs_dropout_prob)
        if use_layer_norm:
            self.layer_norm = nn.LayerNorm(dim_value_all)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
    ):
        from_sequence = hidden_states
        to_sequence = hidden_states

        if encoder_hidden_states is not None:
            to_sequence = encoder_hidden_states
            attention_mask = encoder_attention_mask

        query_layer = self.query(from_sequence)
        key_layer = self.key(to_sequence)
        mixed_query = query_layer[..., None, :, :] * self.m_c[..., :, None, :]
        mixed_key = key_layer[..., None, :, :]* self.m_t[..., :, None, :]
        attention_scores = torch.matmul(mixed_query, mixed_key.transpose(-1, -2))
        content_bias = self.content_bias(to_sequence)
        broadcast_content_bias = content_bias.transpose(-1, -2).unsqueeze(-2)
        attention_scores += broadcast_content_bias
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        attention_probs = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.dropout(attention_probs)

        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        value_layer = self.value(to_sequence)
        value_layer = self.transpose_for_scores(value_layer)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.dim_value_all,)
        context_layer = context_layer.view(*new_context_layer_shape)
        context_layer = self.dense(context_layer)

        if self.use_layer_norm:
            context_layer = self.layer_norm(from_sequence + context_layer)

        if self.output_attentions:
            return (context_layer, attention_probs)
        else:
            return (context_layer,)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, -1)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def init_mixing_matrix(self, scale=0.2):
        mixing = torch.zeros(self.num_attention_heads, self.dim_key_query_all)

        if self.mixing_initialization is MixingMatrixInit.CONCATENATE:

            dim_head = int(math.ceil(self.dim_key_query_all / self.num_attention_heads))
            for i in range(self.num_attention_heads):
                mixing[i, i * dim_head : (i + 1) * dim_head] = 1.0

        elif self.mixing_initialization is MixingMatrixInit.ALL_ONES:
            mixing.fill_(1.0)
            
        elif self.mixing_initialization is MixingMatrixInit.UNIFORM:
            mixing.normal_(std=scale)
        else:
            raise ValueError(
                "Unknown mixing matrix initialization: {}".format(
                    self.mixing_initialization
                )
            )
        return nn.Parameter(mixing)

In [ ]:
class TransformerEncoderLayerWithCollaborativeAttention(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = CollaborativeAttention(
            dim_input=d_model, dim_value_all=d_model, dim_key_query_all=d_model,
            dim_output=d_model, num_attention_heads=num_heads,
            output_attentions=False, attention_probs_dropout_prob=dropout,
            use_dense_layer=True, use_layer_norm=False,  # <-- off here
            mixing_initialization=MixingMatrixInit.UNIFORM,
        )
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),                    # GELU works nicely on ECG
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Pre-LN
        h = x
        x = self.norm1(x)
        x = self.attn(x)[0]
        x = h + self.dropout1(x)
        h = x
        x = self.norm2(x)
        x = self.ff(x)
        x = h + self.dropout2(x)
        return x

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, num_layers: int, num_heads: int, d_model: int, ff_dim: int, embed_mode: str, num_lead: int = 1, chunk_len: int = 50, embedding_dim: int = 512, **kwargs,) -> None:
        super(TransformerModel, self).__init__()
        self.embedding_dim = embedding_dim
        print("Transformer with embedding dim {}".format(self.embedding_dim))

        if embed_mode == "linear":
            self.embed = LinearEmbed(num_lead, chunk_len, d_model)
        elif embed_mode == "cnn":
            self.embed = ConvEmbed(num_lead, d_model)
        else:
            raise NotImplementedError(f"Embedding model `{embed_mode}` is not implemented")

        self.pos_encoder = PositionalEncoding(d_model)

        self.transformer_encoder = nn.Sequential(
            *[TransformerEncoderLayerWithCollaborativeAttention(d_model, num_heads, ff_dim) for _ in range(num_layers)]
        )
        self.fc = nn.Linear(d_model, self.embedding_dim)
        self.name = "transformer"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        feat = self.embed(x)
        feat = torch.clamp(feat, min=-1*clamp_val, max=clamp_val)
        feat = self.pos_encoder(feat)
        feat = self.transformer_encoder(feat)
        feat = feat.permute(0, 2, 1)
        feat = nn.AdaptiveAvgPool1d(1)(feat)
        feat = feat.squeeze(-1)
        feat = self.fc(feat)
        return feat


In [ ]:
def _transformer(
    arch: str,
    num_layers: int,
    num_heads: int,
    d_model: int,
    ff_dim: int,
    embed_mode: str,
    **kwargs: Any,
) -> TransformerModel:
    model = TransformerModel(
        num_layers, num_heads, d_model, ff_dim, embed_mode, **kwargs)
    return model

In [ ]:
def transformer_d8_h4_dim64c(**kwargs: Any) -> TransformerModel:
    num_layers = 4 #Check it on {1,2,3,4,5,6,7}
    num_heads = 4 #Check it on {1,2,3,5,6}
    d_model = 512 #Check on {32,64,128,356,512,768}
    ff_dim = 4*d_model
    embed_mode = "cnn"
    return _transformer("transformer_d8_h4_dim64c", num_layers, num_heads,
                        d_model, ff_dim, embed_mode, **kwargs)

In [ ]:
class SpatialAttention1DCBAM(nn.Module):
    def __init__(self, kernel_size=3):
        super(SpatialAttention1DCBAM, self).__init__()
        assert kernel_size % 2 == 1, "Kernel size must be odd"
        self.conv1 = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """
        x: Tensor of shape (batch, channels, seq_len)
        """
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv1(concat))
        out = x * attention
        return out

In [ ]:
class CBAM1D(nn.Module):
    def __init__(self, n_channels_in):
        super(CBAM1D, self).__init__()

        self.conv1 = nn.Conv1d(n_channels_in, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, n_channels_in, kernel_size=3, padding=1)

        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

        self.spatial_attention = SpatialAttention1DCBAM(kernel_size=3)

    def forward(self, f):
        x = self.relu(self.bn1(self.conv1(f)))   # [B,32,L]
        x = self.dropout1(x)

        x = self.relu(self.bn2(self.conv2(x)))   # [B,64,L]
        x = self.dropout2(x)

        x = self.conv3(x)                        # [B,
        spat_att = self.spatial_attention(x)

        mut_val = spat_att
        return mut_val

In [ ]:
import torch
import torch.nn.functional as F

def simclr_loss_fn(latent_embeddings, temperature=0.1):
    """
    Compute SimCLR (NT-Xent) contrastive loss from latent embeddings.

    Args:
        latent_embeddings (torch.Tensor): shape (2, B, H), two augmented views
        temperature (float): Temperature parameter

    Returns:
        torch.Tensor: Scalar loss value
    """

    z1, z2 = latent_embeddings[0], latent_embeddings[1]  # (B, H), (B, H)
    B = z1.shape[0]

    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)

    sim = torch.mm(z, z.T) / temperature
    mask = torch.eye(2 * B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)

    labels = torch.arange(2 * B, device=z.device)
    labels = (labels + B) % (2 * B)

    loss = F.cross_entropy(sim, labels)
    return loss

## Base Model

In [ ]:
class BaseModel(nn.Module):
    def __init__(self, encoder_name, proj_hidden_dim, output_dim, **kwargs):
        super().__init__()
        self.encoder_name = encoder_name
        self.encoder = transformer_d8_h4_dim64c(**kwargs)

        print("Loaded {} backbone.".format(self.encoder_name))


        self.projector = nn.Sequential(
            nn.Linear(self.encoder.embedding_dim, proj_hidden_dim),
            nn.ReLU(),
            nn.Linear(proj_hidden_dim, output_dim),
        )

    def forward(self, x):
        feats = self.encoder(x)
        z = self.projector(feats)
        return {"feats":feats, "z":z}

In [ ]:
class AdvMaskModel(nn.Module):
    def __init__(self, encoder_name, proj_hidden_dim, output_dim, device, positive_pairing, temperature, **kwargs):

        super().__init__()
        self.base_model = BaseModel(encoder_name=encoder_name, proj_hidden_dim=proj_hidden_dim, output_dim=output_dim,**kwargs)
        self.device = device
        self.temperature = temperature
        self.augmenter = CBAM1D(n_channels_in = 1) 

    def forward(self, x):
        data_lis = [x[:, :, :, j] for j in range(x.shape[-1]) ]
        x_orig = data_lis[0].to(self.device)
        x_transformed = data_lis[1].to(self.device)
        f_masked = self.augmenter(x_transformed)

        z1 = self.base_model(x_orig)["z"]
        z2 = self.base_model(f_masked)["z"]

        nce_loss = simclr_loss_fn([z1, z2], self.temperature)
        return nce_loss

## Traning Loop

In [ ]:
def adv_maks_train(model, train_loader, optimizer, device, epoch, num_epochs, scheduler):
    # set the model to training mode
    model.train()
    simclr_loss = 0

    with tqdm(total=len(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}', unit='batch') as tepoch:
        for idx,sample in enumerate(train_loader):
            optimizer.zero_grad()
            nce_loss = model(sample)
            nce_loss.backward()
            optimizer.step()
            simclr_loss += nce_loss.item()
            tepoch.set_postfix(loss=simclr_loss/(tepoch.n+1))
            tepoch.update(1) 
    scheduler.step()
        
    return simclr_loss/len(train_loader)

In [ ]:
data_path  = "#"

In [ ]:
train_data = DatasetWrapper(data_path=data_path, noise_std=0.1 )  

In [ ]:
batch_size = 64          # 512
num_workers = 4
lr = 0.0001              # 0.0001
weight_decay = 0.05      # 0.01
embedding_dim = 512      # 512
encoder_name = "transformer"
output_dim = 128
proj_hidden_dim = 2048
temperature = 0.07  
num_epochs = 150
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_loader = DataLoader(train_data, batch_size=batch_size,  num_workers=num_workers, shuffle=True)

In [ ]:
model = AdvMaskModel(encoder_name, proj_hidden_dim, output_dim, device, positive_pairing="SimCLR", temperature= temperature)

In [ ]:
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-8)

In [ ]:
save_dir = "saved_models/adv_masking"
os.makedirs(save_dir, exist_ok=True)

best_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
early_stopping_patience = 10  

for epoch in range(num_epochs):
    loss = adv_maks_train(model, train_loader, optimizer, device, epoch, num_epochs, scheduler)


    epoch_save_path = os.path.join(save_dir, f"model_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), epoch_save_path)
    print(f"Saved model for epoch {epoch+1} to {epoch_save_path}")


    if loss < best_loss:
        best_loss = loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0  # reset counter

        best_save_path = os.path.join(save_dir, "best_model.pth")
        torch.save(best_model_state, best_save_path)
        print(f" New best model saved at epoch {epoch+1} with loss {best_loss:.6f}")
    else:
        epochs_no_improve += 1
        print(f" No improvement for {epochs_no_improve} epochs.")

    if epochs_no_improve >= early_stopping_patience:
        print(f"Early stopping triggered after {epoch + 1} epochs.")
        break

    torch.cuda.empty_cache()